In [27]:
# ============================================================
# Ref002 vs Ref004 matching  (Google Colab)
# Blank publishers are treated as a match (wildcard).
# ============================================================
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# ------------------------- CONFIG ---------------------------
CSV_PATH   = '/content/drive/MyDrive/Duplicates_2026_11_11_CombinedData.csv'
OUT_DIR    = '/content/drive/MyDrive'

# Publisher rule: a pair matches if ANY of these is true
#   - Ref002 publisher is blank
#   - Ref004 publisher is blank
#   - Ref002 publisher is a substring of Ref004 publisher
#   - (optional) Ref004 publisher is a substring of Ref002 publisher
BIDIRECTIONAL_PUBLISHER = False

# If True, rows with a blank value in a key field can never match.
# If False, blank == blank counts as equal for key fields.
REQUIRE_NONBLANK_KEYS = False

# Keep 'Title_Full', 'Year', 'Publication_Month', and 'Format' to ignore 'Price_Yuan'
STAGES = {
    'full': dict(keys=['Title_Full', 'Year', 'Publication_Month', 'Format'], sort='Title_Full')
}
# ------------------------------------------------------------


# ---------------------- 1. LOAD -----------------------------
df = pd.read_csv(CSV_PATH, sep=None, engine='python', encoding='utf-8-sig')
print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")

required = {'Source', 'Title_Full', 'Year', 'Publication_Month', 'Format', 'Publisher_1'}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

# ---------------------- 2. SPLIT ----------------------------
src = df['Source'].astype(str)
is002 = src.str.contains('Ref002', na=False, regex=False)
is004 = src.str.contains('Ref004', na=False, regex=False)

ref002_df = df[is002].copy()
ref004_df = df[is004].copy()
ref002_df.index.name = 'idx002'
ref004_df.index.name = 'idx004'

print(f"Ref002 rows: {len(ref002_df)} | Ref004 rows: {len(ref004_df)}")

# ------------------- 3. NORMALISED MATCH COLUMNS ------------
def norm(s, col):
    s = s.fillna('').astype(str).str.strip()
    if col == 'Year' or col == 'Publication_Month':
        s = s.str.replace(r'\.0+$', '', regex=True)
    return s.str.lower().str.replace(r'\s+', ' ', regex=True)

for d in (ref002_df, ref004_df):
    for c in ['Title_Full', 'Year', 'Publication_Month', 'Format', 'Publisher_1']:
        d['_m_' + c] = norm(d[c], c)

# ------------------- 4. PUBLISHER RULE ----------------------
def publisher_match_type(p2, p4):
    if p2 == '' and p4 == '':
        return 'both_blank'
    if p2 == '':
        return 'ref002_blank'
    if p4 == '':
        return 'ref004_blank'
    if p2 in p4:
        return 'substring'
    if BIDIRECTIONAL_PUBLISHER and p4 in p2:
        return 'substring_reversed'
    return None

# ------------------- 5. MATCHING ENGINE ---------------------
def find_pairs(keys):
    mkeys = ['_m_' + k for k in keys]
    left  = ref002_df.reset_index()[['idx002'] + mkeys + ['_m_Publisher_1']]
    right = ref004_df.reset_index()[['idx004'] + mkeys + ['_m_Publisher_1']]

    if REQUIRE_NONBLANK_KEYS:
        left  = left[(left[mkeys] != '').all(axis=1)]
        right = right[(right[mkeys] != '').all(axis=1)]

    pairs = left.merge(right, on=mkeys, suffixes=('_002', '_004'))
    types = [publisher_match_type(a, b)
             for a, b in zip(pairs['_m_Publisher_1_002'], pairs['_m_Publisher_1_004'])]
    pairs['Publisher_Match_Type'] = types
    pairs = pairs[pairs['Publisher_Match_Type'].notna()]
    return pairs[['idx002', 'idx004', 'Publisher_Match_Type']].reset_index(drop=True)

def strip_helpers(d):
    return d[[c for c in d.columns if not c.startswith('_m_')]]

def build_side_by_side(pairs, sort_col):
    a = strip_helpers(ref002_df).loc[pairs['idx002']].add_prefix('Ref002_').reset_index(drop=True)
    b = strip_helpers(ref004_df).loc[pairs['idx004']].add_prefix('Ref004_').reset_index(drop=True)
    out = pd.concat([pairs[['Publisher_Match_Type']].reset_index(drop=True), a, b], axis=1)
    out.insert(0, sort_col, out['Ref002_' + sort_col])
    return out.sort_values(by=sort_col, kind='stable')

def build_vertical(pairs, sort_col):
    base002 = strip_helpers(ref002_df)
    base004 = strip_helpers(ref004_df)

    ids002 = pairs['idx002'].unique()
    r2 = base002.loc[ids002].copy()
    r2['Match_Group'] = [f"Match_{i}" for i in ids002]
    r2['Record_Source'] = 'Ref002'
    r2['Publisher_Match_Type'] = ''
    r2['_grp'] = ids002
    r2['_ord'] = 0

    r4 = base004.loc[pairs['idx004']].copy()
    r4['Match_Group'] = [f"Match_{i}" for i in pairs['idx002']]
    r4['Record_Source'] = 'Ref004'
    r4['Publisher_Match_Type'] = pairs['Publisher_Match_Type'].values
    r4['_grp'] = pairs['idx002'].values
    r4['_ord'] = 1

    out = pd.concat([r2, r4], ignore_index=True)
    sort_vals = base002[sort_col].astype(str)
    out['_sort'] = out['_grp'].map(sort_vals)
    out = out.sort_values(['_sort', '_grp', '_ord'], kind='stable')
    out = out.drop(columns=['_grp', '_ord', '_sort'])

    front = ['Match_Group', 'Record_Source', 'Publisher_Match_Type',
             'Title_Full', 'Year', 'Publication_Month', 'Format', 'Publisher_1']
    front = [c for c in front if c in out.columns]
    return out[front + [c for c in out.columns if c not in front]]

# ------------------- 6. RUN STAGE --------------------------
for name, cfg in STAGES.items():
    keys, sort_col = cfg['keys'], cfg['sort']
    pairs = find_pairs(keys)

    matched_ids = pairs['idx002'].unique()
    unmatched_df = strip_helpers(ref002_df).loc[~ref002_df.index.isin(matched_ids)]

    side_df = build_side_by_side(pairs, sort_col) if len(pairs) else pd.DataFrame()
    vert_df = build_vertical(pairs, sort_col) if len(pairs) else pd.DataFrame()

    unmatched_df.to_csv(f'{OUT_DIR}/Unmatched_Ref002_{name}.csv', index=False, encoding='utf-8-sig')
    if len(pairs):
        side_df.to_csv(f'{OUT_DIR}/Matched_Ref002_Ref004_AllColumns_{name}.csv', index=False, encoding='utf-8-sig')
        vert_df.to_csv(f'{OUT_DIR}/Matched_Ref002_Ref004_Vertical_{name}.csv', index=False, encoding='utf-8-sig')

    uniq_titles = ref002_df.loc[matched_ids, sort_col].nunique() if len(matched_ids) else 0
    by_type = pairs['Publisher_Match_Type'].value_counts().to_dict()

    print(f"\n=== Stage '{name}'  keys={keys} (+ publisher rule) ===")
    print(f"Ref002 records:             {len(ref002_df)}")
    print(f"  matched (unique records): {len(matched_ids)}")
    print(f"  unmatched:                {len(unmatched_df)}")
    print(f"Ref002-Ref004 pairs:        {len(pairs)}")
    print(f"Unique {sort_col} matched:  {uniq_titles}")
    print(f"Publisher match types:      {by_type}")
    if len(pairs):
        display(vert_df.head(10))

print(f"\nFiles saved to: {OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded: 38162 rows, 47 columns
Ref002 rows: 2796 | Ref004 rows: 35362

=== Stage 'full'  keys=['Title_Full', 'Year', 'Publication_Month', 'Format'] (+ publisher rule) ===
Ref002 records:             2796
  matched (unique records): 1554
  unmatched:                1242
Ref002-Ref004 pairs:        1555
Unique Title_Full matched:  1372
Publisher match types:      {'substring': 1555}


,Match_Group,Record_Source,Publisher_Match_Type,Title_Full,Year,Publication_Month,Format,Publisher_1,Title_Core,Title_Extra,...,Market_Exchange_Price_10品_in_Yuan_in_Year_2001,Page_Count,Additional_Info,Editorial_Comments,Source,Category/Chapter,Footnotes,OCR_Data,OCR_Process,Unnamed: 46
30,Match_35414,Ref002,,80天环游地球,1984.0,5,64,浙江人民美术,80天环游地球,NaN,...,40,NaN,NaN,NaN,Ref002:7,单册：贰划,NaN,80天环游地球/出版社：浙江人民美术/改编：韩幼文/绘画：黄云松张昌询/84年5月1版1次6...,PaddlePaddle,R002_00049
1584,Match_35414,Ref004,substring,80天环游地球,1984.0,5,64,浙江人民美术出版社,80天环游地球,NaN,...,NaN,NaN,NaN,NaN,Ref004:775,NaN,NaN,80天环游地球 韩幼文改编、黄云松、张昌洵绘/浙江人美/5月/64/0.19元,PaddlePaddle,R004_29121
1181,Match_37222,Ref002,,X光与伦琴,1983.0,2,60,人民美术,X光与伦琴,NaN,...,160,NaN,NaN,NaN,Ref002:233,套书：古典题材,NaN,X光与伦琴/出版社：人民美术/编文：范若由/绘画：聂崇瑞/83年2月1版1次60开/印数：7...,PaddlePaddle,R002_01857
2735,Match_37222,Ref004,substring,X光与伦琴,1983.0,2,60,人民美术出版社,X光与伦琴,NaN,...,NaN,NaN,NaN,NaN,Ref004:692,NaN,NaN,X光与伦琴 范若由著、聂崇瑞绘/人美/2月/60/0.13元,PaddlePaddle,R004_26111
209,Match_35691,Ref002,,“支支”历险记,1985.0,4,64,浙江人民美术,“支支”历险记,NaN,...,10,NaN,NaN,NaN,Ref002:41,单册：肆划,NaN,“支支”历险记/出版社：浙江人民美术/改编：辽洋胡霜/绘画：罗枫/85年4月1版1次64开/...,PaddlePaddle,R002_00326
1763,Match_35691,Ref004,substring,“支支”历险记,1985.0,4,64,浙江人民美术出版社,“支支”历险记,NaN,...,NaN,NaN,NaN,NaN,Ref004:854,NaN,NaN,“支支”历险记 辽洋、胡霜编、罗枫绘/浙江人美/4月/64/0.19元,PaddlePaddle,R004_31984
845,Match_36670,Ref002,,“海怪”之谜,1984.0,10,60,人民美术,“海怪”之谜,NaN,...,15,NaN,NaN,NaN,Ref002:164,单册：拾划,《“海怪”之谜》与《海底两万里》为同一题材。,“海怪”之谜/出版社：人民美术/改编：胡源/绘画：毛文彪、吴海燕/84年10月1版1次60开...,PaddlePaddle,R002_01305
2399,Match_36670,Ref004,substring,“海怪”之谜,1984.0,10,60,人民美术出版社,“海怪”之谜,NaN,...,NaN,NaN,NaN,NaN,Ref004:746,NaN,NaN,“海怪”之谜 胡源改编、毛文彪、吴海燕绘/人美/10月/60/0.42元,PaddlePaddle,R004_28080
1054,Match_36993,Ref002,,“镢柄”相亲记,1983.0,1,64,上海人民美术,“镢柄”相亲记,NaN,...,15,NaN,NaN,NaN,Ref002:204,单册：拾肆划~贰拾壹划,NaN,“镢柄”相亲记/出版社：上海人民美术/改编：刘思成/绘画：刘国辉/83年1月1版1次64开/...,PaddlePaddle,R002_01628
2608,Match_36993,Ref004,substring,“镢柄”相亲记,1983.0,1,64,上海人民美术出版社,“镢柄”相亲记,NaN,...,NaN,NaN,NaN,NaN,Ref004:707,NaN,NaN,“镢柄”相亲记 张石山原著、刘思成改编、刘国辉绘/上海人美/1月/64/0.13元,PaddlePaddle,R004_26653



Files saved to: /content/drive/MyDrive


In [28]:
import pandas as pd

# Load original dataset
original_df = pd.read_csv('/content/drive/MyDrive/Duplicates_2026_11_11_CombinedData.csv', sep=None, engine='python', encoding='utf-8-sig')

# Create lookup dictionaries from pairs
ref002_to_ref004 = {}
ref004_to_ref002 = {}
for idx, row in pairs.iterrows():
    r2, r4 = row['idx002'], row['idx004']
    if r2 not in ref002_to_ref004:
        ref002_to_ref004[r2] = []
    ref002_to_ref004[r2].append(r4)

    if r4 not in ref004_to_ref002:
        ref004_to_ref002[r4] = []
    ref004_to_ref002[r4].append(r2)

# Generate sequential IDs based on the original dataframe row order
unique_ids = [None] * len(original_df)
current_id_num = 1

for idx, row in original_df.iterrows():
    # If this row already received a matched group ID from a previously evaluated partner row, skip
    if unique_ids[idx] is not None:
        continue

    # Determine group ID for this entry
    group_id = f"LHH{current_id_num:06d}"
    current_id_num += 1

    unique_ids[idx] = group_id

    # If it is a match, cascade the same ID to all its matched partners
    if idx in ref002_to_ref004:
        for partner_idx in ref002_to_ref004[idx]:
            unique_ids[partner_idx] = group_id
    elif idx in ref004_to_ref002:
        for partner_idx in ref004_to_ref002[idx]:
            unique_ids[partner_idx] = group_id

# Add the column to the original dataset
original_df['Unique_Match_ID'] = unique_ids

# Save the updated CSV file
output_csv_path = '/content/drive/MyDrive/UniqueID_2026_11_11.csv'
original_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"Successfully saved the updated dataset to {output_csv_path}")
print(f"Total rows: {len(original_df)}")
print(f"Unique IDs generated (formatted as LHHXXXXXX): {original_df['Unique_Match_ID'].nunique()}")
display(original_df[['Unique_Match_ID', 'Source', 'Title_Full', 'Year', 'Publication_Month', 'Format', 'Publisher_1']].head(15))

Successfully saved the updated dataset to /content/drive/MyDrive/UniqueID_2026_11_11.csv
Total rows: 38162
Unique IDs generated (formatted as LHHXXXXXX): 36608


,Unique_Match_ID,Source,Title_Full,Year,Publication_Month,Format,Publisher_1
0,LHH000001,Ref004:1,二万五千里长征,1949.0,11,NaN,广益
1,LHH000002,Ref004:1,人民英雄刘志丹,1949.0,12,NaN,大众美术出版社
2,LHH000003,Ref004:1,三打祝家庄,1949.0,NaN,NaN,民众书局
3,LHH000004,Ref004:1,三上轿,1949.0,NaN,NaN,民众书局
4,LHH000005,Ref004:1,工人的旗帜赵占魁,1949.0,11,NaN,大众美术出版社
5,LHH000006,Ref004:1,小姑贤刘保堂,1949.0,10,NaN,晨光出版公司
6,LHH000007,Ref004:1,与灾荒搏斗,1949.0,12,NaN,教育出版社
7,LHH000008,Ref004:1,王秀鸾,1949.0,11,NaN,华东书店
8,LHH000009,Ref004:1,丹娘,1949.0,10,NaN,大众美术出版社
9,LHH000010,Ref004:1,无坚不摧,1949.0,10,NaN,中南新华
